# Convert a Bangla speech model for Kotha

This notebook turns a Bangla fine-tuned Whisper model into the browser format Kotha uses and uploads it to your Hugging Face account. It runs on a free Colab or Kaggle **CPU**; no GPU is needed. It takes about 10–15 minutes.

1. Run the cells in order.
2. When asked, paste a Hugging Face token with **write** access (https://huggingface.co/settings/tokens).
3. Open Kotha with `?model=YOUR_NAME/bangla-asr-onnx` at the end of the URL, or build it with `VITE_BANGLA_MODEL_ID=YOUR_NAME/bangla-asr-onnx`.

In [ ]:
# 1. Get the code and install the converter's dependencies
!git clone --depth 1 -b claude/digital-product-research-k006rr https://github.com/Imtiaj-Sajin/cxr.git kotha
%cd kotha
!pip install -q -r tools/requirements.txt

In [ ]:
# 2. Log in to Hugging Face (needs a token with write access)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Choose the source model and your target repo
SOURCE_MODEL = 'bangla-speech-processing/BanglaASR'  # whisper-small fine-tuned on Bangla
BASE_MODEL = 'openai/whisper-small'                   # the model SOURCE_MODEL was fine-tuned from
TARGET_REPO = 'YOUR_HF_USERNAME/bangla-asr-onnx'       # <- change this

In [ ]:
# 4. Convert and upload
!python tools/convert_model.py --model {SOURCE_MODEL} --base {BASE_MODEL} --output out/model --push {TARGET_REPO} --overwrite

## Optional: measure accuracy

This needs Node.js 20+ (Colab and Kaggle usually have it). It runs the converted model with exactly the same pipeline as the website on the public FLEURS Bangla test set and prints the character error rate (CER) and word error rate (WER). Lower is better.

In [ ]:
!python tools/prepare_eval_data.py --out eval-data/fleurs-bn --count 50
!npm ci --silent
!npx tsx scripts/evaluate.ts --model out/model --data eval-data/fleurs-bn --out eval-data/result.json

### Compare with the plain Whisper model

Run this to see how much the Bangla fine-tune helps (it downloads `onnx-community/whisper-small`).

In [ ]:
!npx tsx scripts/evaluate.ts --model onnx-community/whisper-small --data eval-data/fleurs-bn --out eval-data/result-base.json